In [1]:
# Final Analysis

# This notebook integrates the frozen evaluation results across the four
# longitudinal clinical summarization workflows:

# 1. Direct summarization
# 2. Hierarchical summarization
# 3. RAG summarization
# 4. RAG + Verification

# The notebook performs cross-workflow analysis, investigates mechanisms
# underlying observed differences, and generates paper-ready tables and figures.

# No summaries or primary evaluation judgments are regenerated in this notebook.
# All analyses use previously saved experimental artifacts.

In [2]:
# ============================================================
# SETUP
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
)

EVALUATION_DIR = (
    RESULTS_DIR
    / "evaluation"
)

print("Project root:", PROJECT_ROOT)
print("Evaluation directory:", EVALUATION_DIR)

Project root: /Users/pallavi_chandanshive/projects/clinical-summarization-eval
Evaluation directory: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation


In [5]:
# ============================================================
# LOAD FROZEN EVALUATION RESULTS
#
# These are existing outputs from evaluation notebooks 01–04.
# No evaluation is rerun here.
# ============================================================

# TF-IDF
tfidf_scores = pd.read_csv(
    EVALUATION_DIR / "tfidf_patient_scores.csv"
)

# Completeness
with open(
    EVALUATION_DIR / "completeness_results.json",
    "r",
    encoding="utf-8",
) as f:
    completeness_results = json.load(f)

# Temporal consistency
# IMPORTANT: this is the final valid temporal artifact.
with open(
    EVALUATION_DIR / "temporal_final_order_results.json",
    "r",
    encoding="utf-8",
) as f:
    temporal_results = json.load(f)

# Faithfulness
with open(
    EVALUATION_DIR / "faithfulness_results.json",
    "r",
    encoding="utf-8",
) as f:
    faithfulness_results = json.load(f)


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

print("TF-IDF rows:", len(tfidf_scores))
print("Completeness patients:", len(completeness_results))
print("Temporal patients:", len(temporal_results))
print("Faithfulness patients:", len(faithfulness_results))

TF-IDF rows: 200
Completeness patients: 50
Temporal patients: 50
Faithfulness patients: 50


In [6]:
# ============================================================
# INSPECT SAVED RAG ARTIFACTS
#
# Goal:
# Determine whether the exact Top-20 evidence used for final
# RAG summarization was saved.
# ============================================================

RAG_DIR = RESULTS_DIR / "rag"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("RAG result files:")
for path in sorted(RAG_DIR.glob("*")):
    if path.is_file():
        print(" -", path.name)

print("\nProcessed files potentially related to RAG:")
for path in sorted(PROCESSED_DIR.glob("*rag*")):
    if path.is_file():
        print(" -", path.name)

RAG result files:
 - final_rag_summaries.json

Processed files potentially related to RAG:
 - rag_atomic_claims.json
 - rag_verification_bge_retrievals.json


In [7]:
# ============================================================
# LOCATE RAG IMPLEMENTATION NOTEBOOKS
# ============================================================

RAG_NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "rag"

for path in sorted(RAG_NOTEBOOK_DIR.glob("*.ipynb")):
    print(path.name)

01_configuration_summaries.ipynb
02_configuration_evaluation.ipynb
03_rag_implementation.ipynb


In [8]:
# ============================================================
# INSPECT SAVED WORKFLOW-LEVEL SUMMARY FILES
#
# These summaries were produced by the completed evaluation
# notebooks. No scores are recalculated here.
# ============================================================

print("Saved workflow-level summary files:\n")

for path in sorted(EVALUATION_DIR.glob("*summary*.csv")):
    print(path.name)

    df = pd.read_csv(path)

    print("Columns:", list(df.columns))
    print(df)
    print()

Saved workflow-level summary files:

faithfulness_label_summary.csv
Columns: ['workflow', 'total_claims', 'supported', 'partially_supported', 'unsupported', 'supported_pct', 'partially_supported_pct', 'unsupported_pct']
           workflow  total_claims  supported  partially_supported  \
0            direct          3000       2969                   23   
1      hierarchical          2905       2879                   19   
2               rag          2658       2603                   21   
3  rag_verification          2407       2366                   14   

   unsupported  supported_pct  partially_supported_pct  unsupported_pct  
0            8      98.966667                 0.766667         0.266667  
1            7      99.104991                 0.654045         0.240964  
2           34      97.930775                 0.790068         1.279157  
3           27      98.296635                 0.581637         1.121728  

faithfulness_workflow_summary.csv
Columns: ['workflow', 'mean',

In [9]:
# ============================================================
# BUILD FINAL CROSS-METRIC RESULTS MATRIX
#
# Uses only frozen evaluation results.
# No LLM calls and no evaluation is rerun.
# ============================================================

# ------------------------------------------------------------
# 1. Completeness means from frozen fact-level judgments
# ------------------------------------------------------------

completeness_rows = []

for person_id, result in completeness_results.items():

    scores = result["scores"]
    n_facts = len(scores)

    for workflow in [
        "direct",
        "hierarchical",
        "rag",
        "rag_verification",
    ]:

        total_points = sum(
            fact[workflow]
            for fact in scores
        )

        score = (
            total_points
            / (2 * n_facts)
            * 100
        )

        completeness_rows.append({
            "person_id": person_id,
            "workflow": workflow,
            "score": score,
        })

completeness_df = pd.DataFrame(completeness_rows)

completeness_summary = (
    completeness_df
    .groupby("workflow")["score"]
    .mean()
)


# ------------------------------------------------------------
# 2. Load already-saved workflow summaries
# ------------------------------------------------------------

faithfulness_summary = pd.read_csv(
    EVALUATION_DIR / "faithfulness_workflow_summary.csv"
).set_index("workflow")["mean"]

temporal_summary = pd.read_csv(
    EVALUATION_DIR / "temporal_workflow_summary.csv"
).set_index("workflow")["mean"]

tfidf_summary = pd.read_csv(
    EVALUATION_DIR / "tfidf_workflow_summary.csv"
).set_index("workflow")["mean"]


# ------------------------------------------------------------
# 3. Combine all four metrics
# ------------------------------------------------------------

workflow_order = [
    "direct",
    "hierarchical",
    "rag",
    "rag_verification",
]

final_matrix = pd.DataFrame({
    "Completeness (%)": completeness_summary,
    "Faithfulness (%)": faithfulness_summary,
    "Temporal consistency (%)": temporal_summary,
    "TF-IDF cosine": tfidf_summary,
}).loc[workflow_order]


# Friendly labels for presentation
final_matrix.index = [
    "Direct",
    "Hierarchical",
    "RAG",
    "RAG + Verification",
]


# ------------------------------------------------------------
# Display final matrix
# ------------------------------------------------------------

display(
    final_matrix.round({
        "Completeness (%)": 2,
        "Faithfulness (%)": 2,
        "Temporal consistency (%)": 2,
        "TF-IDF cosine": 3,
    })
)

,Completeness (%),Faithfulness (%),Temporal consistency (%),TF-IDF cosine
Direct,99.28,99.36,98.98,0.548
Hierarchical,98.30,99.53,98.83,0.537
RAG,87.58,98.18,98.91,0.523
RAG + Verification,85.52,98.54,98.69,0.521


In [10]:
# ============================================================
# OMNIBUS STATISTICAL RESULTS
#
# These values come from the completed evaluation notebooks.
# They are recorded here for integrated interpretation only.
# ============================================================

statistical_results = pd.DataFrame({
    "Metric": [
        "Completeness",
        "Faithfulness",
        "Temporal consistency",
        "TF-IDF cosine",
    ],
    "Friedman statistic": [
        97.7298,
        1.515152,
        0.563218,
        34.3636,
    ],
    "p_value": [
        4.78e-21,
        0.678778,
        0.904799,
        np.nan,  # replace only if exact saved p-value is available
    ],
})

display(statistical_results)

,Metric,Friedman statistic,p_value
0,Completeness,97.729800,4.780000e-21
1,Faithfulness,1.515152,6.787780e-01
2,Temporal consistency,0.563218,9.047990e-01
3,TF-IDF cosine,34.363600,NaN


In [11]:
# ============================================================
# COMPLETENESS DIFFERENCES
# ============================================================

completeness_means = final_matrix["Completeness (%)"]

comparisons = [
    ("Direct", "Hierarchical"),
    ("Direct", "RAG"),
    ("Direct", "RAG + Verification"),
    ("Hierarchical", "RAG"),
    ("Hierarchical", "RAG + Verification"),
    ("RAG", "RAG + Verification"),
]

difference_rows = []

for workflow_a, workflow_b in comparisons:

    difference = (
        completeness_means[workflow_a]
        - completeness_means[workflow_b]
    )

    difference_rows.append({
        "Comparison": f"{workflow_a} vs {workflow_b}",
        "Difference (percentage points)": difference,
    })

completeness_differences = pd.DataFrame(difference_rows)

display(
    completeness_differences.round(2)
)

,Comparison,Difference (percentage points)
0,Direct vs Hierarchical,0.98
1,Direct vs RAG,11.70
2,Direct vs RAG + Verification,13.76
3,Hierarchical vs RAG,10.72
4,Hierarchical vs RAG + Verification,12.78
5,RAG vs RAG + Verification,2.06


In [12]:
# ============================================================
# COMPLETENESS PAIRWISE STATISTICAL RESULTS
#
# Previously computed in the completeness evaluation.
# No statistical tests are rerun here.
# ============================================================

completeness_pairwise = pd.DataFrame({
    "Comparison": [
        "Direct vs Hierarchical",
        "Direct vs RAG",
        "Direct vs RAG + Verification",
        "Hierarchical vs RAG",
        "Hierarchical vs RAG + Verification",
        "RAG vs RAG + Verification",
    ],
    "Difference (pp)": [
        0.98,
        11.70,
        13.76,
        10.72,
        12.78,
        2.06,
    ],
    "Holm-adjusted p": [
        0.007504288,
        5.996271e-08,
        2.555148e-08,
        1.847389e-07,
        2.555148e-08,
        1.397505e-04,
    ],
})

completeness_pairwise["Significant"] = (
    completeness_pairwise["Holm-adjusted p"] < 0.05
)

display(completeness_pairwise)

,Comparison,Difference (pp),Holm-adjusted p,Significant
0,Direct vs Hierarchical,0.98,7.504288e-03,True
1,Direct vs RAG,11.70,5.996271e-08,True
2,Direct vs RAG + Verification,13.76,2.555148e-08,True
3,Hierarchical vs RAG,10.72,1.847389e-07,True
4,Hierarchical vs RAG + Verification,12.78,2.555148e-08,True
5,RAG vs RAG + Verification,2.06,1.397505e-04,True


In [13]:
# ============================================================
# FAITHFULNESS + TEMPORAL CONSISTENCY OVERVIEW
#
# Omnibus tests were not significant, so no post-hoc
# pairwise comparisons are performed.
# ============================================================

stable_metrics = pd.DataFrame({
    "Metric": [
        "Faithfulness",
        "Temporal consistency",
    ],
    "Lowest workflow mean (%)": [
        final_matrix["Faithfulness (%)"].min(),
        final_matrix["Temporal consistency (%)"].min(),
    ],
    "Highest workflow mean (%)": [
        final_matrix["Faithfulness (%)"].max(),
        final_matrix["Temporal consistency (%)"].max(),
    ],
    "Range (pp)": [
        final_matrix["Faithfulness (%)"].max()
        - final_matrix["Faithfulness (%)"].min(),

        final_matrix["Temporal consistency (%)"].max()
        - final_matrix["Temporal consistency (%)"].min(),
    ],
    "Friedman p": [
        0.678778,
        0.904799,
    ],
})

display(stable_metrics.round(3))

,Metric,Lowest workflow mean (%),Highest workflow mean (%),Range (pp),Friedman p
0,Faithfulness,98.183,99.527,1.344,0.679
1,Temporal consistency,98.690,98.980,0.290,0.905


In [ ]:
### Integrated Interpretation

# The four workflow architectures differed primarily in completeness rather than
# faithfulness or temporal consistency.

# Completeness showed a significant workflow effect, with Direct (99.28%) and
# Hierarchical (98.30%) summarization retaining substantially more clinically
# important information than the tested RAG (87.58%) and RAG + Verification
# (85.52%) workflows. All pairwise completeness comparisons remained significant
# after Holm correction, although the Direct–Hierarchical difference was small
# in magnitude (0.98 percentage points).

# In contrast, faithfulness remained high across all workflows (98.18–99.53%)
# and did not differ significantly across architectures (Friedman p = .679).
# Temporal consistency was similarly high (98.69–98.98%), with no significant
# workflow effect (Friedman p = .905).

# Together, these results suggest that, under the evaluated configurations,
# workflow architecture had a substantially greater effect on the amount of
# clinically important information retained in the final summary than on the
# faithfulness or temporal ordering of information that was represented.

In [14]:
# ============================================================
# RAG → RAG + VERIFICATION:
# IDENTIFY COMPLETENESS LOSSES
#
# A loss occurs when a clinically important reference fact
# received a lower completeness score after verification.
# ============================================================

verification_loss_rows = []

for person_id, result in completeness_results.items():

    for fact in result["scores"]:

        rag_score = fact["rag"]
        verified_score = fact["rag_verification"]

        # Keep only facts whose completeness decreased
        if verified_score < rag_score:

            verification_loss_rows.append({
                "person_id": person_id,
                "fact_id": fact["fact_id"],
                "rag_score": rag_score,
                "verified_score": verified_score,
                "score_change": verified_score - rag_score,
            })

verification_losses = pd.DataFrame(verification_loss_rows)

print(
    "Number of reference facts with lower completeness after verification:",
    len(verification_losses)
)

print(
    "Number of affected patients:",
    verification_losses["person_id"].nunique()
)

print("\nTypes of score decreases:")

display(
    verification_losses
    .groupby(["rag_score", "verified_score"])
    .size()
    .reset_index(name="count")
)

display(verification_losses.head(10))

Number of reference facts with lower completeness after verification: 28
Number of affected patients: 16

Types of score decreases:


,rag_score,verified_score,count
0,1,0,3
1,2,0,3
2,2,1,22


,person_id,fact_id,rag_score,verified_score,score_change
0,05192757-942f-460d-b4ff-004ec39cc5ee,3,2,1,-1
1,05192757-942f-460d-b4ff-004ec39cc5ee,8,2,1,-1
2,05192757-942f-460d-b4ff-004ec39cc5ee,10,2,1,-1
3,05192757-942f-460d-b4ff-004ec39cc5ee,14,2,1,-1
4,05192757-942f-460d-b4ff-004ec39cc5ee,20,2,1,-1
5,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,12,2,1,-1
6,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,13,2,1,-1
7,29ea304f-821d-474e-81a1-394ca3945e02,7,2,1,-1
8,31f9612b-5a6b-48ea-887b-895772a83b99,20,2,0,-2
9,359014a1-10e6-4bd8-9ba7-513d021c971e,8,2,1,-1


In [15]:
# ============================================================
# INSPECT COMPLETENESS REFERENCE CHECKLIST STRUCTURE
# ============================================================

with open(
    EVALUATION_DIR / "completeness_reference_checklists.json",
    "r",
    encoding="utf-8",
) as f:
    completeness_references = json.load(f)

# Inspect one patient so we know the exact saved structure
first_patient_id = next(iter(completeness_references))

print("Patient:", first_patient_id)
print(json.dumps(
    completeness_references[first_patient_id],
    indent=2
)[:5000])

Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
{
  "person_id": "028998ee-babc-4096-9b28-001bc2f9a84e",
  "facts": [
    {
      "fact_id": 1,
      "fact": "15-year-old male presented to the ED with a 3-day history of severe lower abdominal pain and constipation.",
      "category": "diagnosis"
    },
    {
      "fact_id": 2,
      "fact": "Initial ED impression was constipation, later associated with mild dehydration.",
      "category": "diagnosis"
    },
    {
      "fact_id": 3,
      "fact": "Abdominal X-ray showed significant faecal loading with no evidence of obstruction or perforation.",
      "category": "investigation"
    },
    {
      "fact_id": 4,
      "fact": "Blood tests including FBC, U&Es, CRP, and LFTs were normal.",
      "category": "investigation"
    },
    {
      "fact_id": 5,
      "fact": "The patient was treated with IV 0.9% sodium chloride for mild dehydration.",
      "category": "treatment"
    },
    {
      "fact_id": 6,
      "fact": "A single 10 mg 

In [16]:
# ============================================================
# ATTACH CLINICAL FACT TEXT TO COMPLETENESS LOSSES
# ============================================================

loss_details = []

for _, row in verification_losses.iterrows():

    person_id = row["person_id"]
    fact_id = row["fact_id"]

    # Get this patient's reference checklist
    patient_reference = completeness_references[person_id]

    # Find the matching clinical fact
    matching_fact = next(
        fact
        for fact in patient_reference["facts"]
        if fact["fact_id"] == fact_id
    )

    loss_details.append({
        "person_id": person_id,
        "fact_id": fact_id,
        "category": matching_fact["category"],
        "fact": matching_fact["fact"],
        "rag_score": row["rag_score"],
        "verified_score": row["verified_score"],
        "score_change": row["score_change"],
    })

verification_loss_details = pd.DataFrame(loss_details)

print("Completeness losses:", len(verification_loss_details))
print(
    "Affected patients:",
    verification_loss_details["person_id"].nunique()
)

display(
    verification_loss_details[
        [
            "person_id",
            "fact_id",
            "category",
            "fact",
            "rag_score",
            "verified_score",
        ]
    ]
)

Completeness losses: 28
Affected patients: 16


,person_id,fact_id,category,fact,rag_score,verified_score
0,05192757-942f-460d-b4ff-004ec39cc5ee,3,diagnosis,Pre-operative assessment showed mild anaemia w...,2,1
1,05192757-942f-460d-b4ff-004ec39cc5ee,8,procedure,The removed left femoral head was sent for his...,2,1
2,05192757-942f-460d-b4ff-004ec39cc5ee,10,complication,Estimated intraoperative blood loss was approx...,2,1
3,05192757-942f-460d-b4ff-004ec39cc5ee,14,complication,"Postoperative anaemia developed after surgery,...",2,1
4,05192757-942f-460d-b4ff-004ec39cc5ee,20,treatment,Discharge medications included rivaroxaban 10 ...,2,1
5,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,12,clinical_course,The patient mobilised progressively after surg...,2,1
6,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,13,clinical_course,Occupational therapy identified the need for h...,2,1
7,29ea304f-821d-474e-81a1-394ca3945e02,7,investigation,Repeat chest imaging showed gradual improvemen...,2,1
8,31f9612b-5a6b-48ea-887b-895772a83b99,20,treatment,Occupational therapy recommended crutches and ...,2,0
9,359014a1-10e6-4bd8-9ba7-513d021c971e,8,clinical_course,Serial chest radiographs showed progressive im...,2,1


In [17]:
# ============================================================
# LOAD ORIGINAL RAG + FINAL VERIFIED SUMMARIES
# ============================================================

with open(
    RESULTS_DIR / "rag" / "final_rag_summaries.json",
    "r",
    encoding="utf-8",
) as f:
    rag_summaries = json.load(f)

with open(
    RESULTS_DIR / "rag_verification" / "final_verified_summaries.json",
    "r",
    encoding="utf-8",
) as f:
    verified_summaries = json.load(f)


# ------------------------------------------------------------
# Inspect the first affected patient
# ------------------------------------------------------------

first_loss = verification_loss_details.iloc[0]

person_id = first_loss["person_id"]
fact_id = first_loss["fact_id"]

print("PERSON:", person_id)
print("FACT ID:", fact_id)
print("\nREFERENCE FACT:")
print(first_loss["fact"])

print("\n" + "=" * 80)
print("ORIGINAL RAG SUMMARY:")
print(rag_summaries[person_id]["summary"])

print("\n" + "=" * 80)
print("VERIFIED SUMMARY:")
print(verified_summaries[person_id]["final_summary"])

PERSON: 05192757-942f-460d-b4ff-004ec39cc5ee
FACT ID: 3

REFERENCE FACT:
Pre-operative assessment showed mild anaemia with hemoglobin 11.3 g/dL, and ferrous sulfate was started for optimization before surgery.

ORIGINAL RAG SUMMARY:
- **21/12/2025 – Preoperative assessment:** Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis, associated with stiffness and limited mobility. Medical history included hypertension, severe left hip OA, and cataract surgery in 2018. BP was 145/85 mmHg and Hb 11.3 g/dL, consistent with mild anaemia; she was taking ferrous sulfate 200 mg daily, paracetamol as needed, and calcium/vitamin D supplements. ECG showed normal sinus rhythm without ischaemic changes. No allergies were reported. She was deemed suitable for spinal anaesthesia, consented for elective total left hip replacement, and cleared for admission on 02/01/2026.

- **02/01/2026 – Admission and surgery:** Preoperative checks confirmed identity, fa

In [18]:
# ============================================================
# INSPECT WORKFLOW 4 VERIFICATION ARTIFACT
# ============================================================

VERIFICATION_DIR = RESULTS_DIR / "rag_verification"

print("Workflow 4 result files:\n")

for path in sorted(VERIFICATION_DIR.glob("*")):
    if path.is_file():
        print(" -", path.name)

Workflow 4 result files:

 - final_verified_summaries.json
 - rag_verification_results.json


In [19]:
# ============================================================
# INSPECT SAVED VERIFICATION DECISIONS FOR FIRST CASE
# ============================================================

with open(
    RESULTS_DIR / "rag_verification" / "rag_verification_results.json",
    "r",
    encoding="utf-8",
) as f:
    verification_results = json.load(f)

person_id = "05192757-942f-460d-b4ff-004ec39cc5ee"

print(
    json.dumps(
        verification_results[person_id],
        indent=2
    )[:12000]
)

{
  "person_id": "05192757-942f-460d-b4ff-004ec39cc5ee",
  "source_group_count": 29,
  "source_groups": {
    "1": {
      "source_id": 1,
      "source_sentence": "**21/12/2025 \u2013 Preoperative assessment:** Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis, associated with stiffness and limited mobility.",
      "verified_claims": [
        {
          "claim_id": 1,
          "claim": "Hope Chinwo was evaluated.",
          "source_id": 1,
          "source_sentence": "**21/12/2025 \u2013 Preoperative assessment:** Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis, associated with stiffness and limited mobility.",
          "label": "SUPPORTED",
          "reason": "The record documents pre-op assessment/checks and notes the patient was seen/evaluated on 21/12/25.",
          "supporting_evidence_ranks": [
            1,
            2,
            5
          ]
        },
 

In [20]:
# ============================================================
# INSPECT VERIFICATION RESULT STRUCTURE
# ============================================================

# Use the same affected patient we already examined
person_id = "05192757-942f-460d-b4ff-004ec39cc5ee"

patient_verification = verification_results[person_id]

print("Top-level keys:")
print(patient_verification.keys())

print("\nNumber of source groups:")
print(len(patient_verification["source_groups"]))

# Collect all verification labels for this patient
label_counts = {}

for source_group in patient_verification["source_groups"].values():

    for claim in source_group["verified_claims"]:
        label = claim["label"]

        label_counts[label] = label_counts.get(label, 0) + 1

print("\nVerification label counts:")
print(label_counts)

# Show every claim that triggered verification intervention
print("\nNON-FULLY-SUPPORTED CLAIMS:\n")

for source_group in patient_verification["source_groups"].values():

    for claim in source_group["verified_claims"]:

        if claim["label"] != "SUPPORTED":

            print("Claim ID:", claim["claim_id"])
            print("Claim:", claim["claim"])
            print("Label:", claim["label"])
            print("Reason:", claim["reason"])
            print("-" * 80)

Top-level keys:
dict_keys(['person_id', 'source_group_count', 'source_groups'])

Number of source groups:
29

Verification label counts:
{'SUPPORTED': 115, 'UNSUPPORTED': 8, 'PARTIALLY_SUPPORTED': 1}

NON-FULLY-SUPPORTED CLAIMS:

Claim ID: 3
Claim: Hope Chinwo's left hip pain was gradually worsening.
Label: UNSUPPORTED
Reason: The record supports severe left hip OA and limited mobility, but does not explicitly state the pain was gradually worsening.
--------------------------------------------------------------------------------
Claim ID: 5
Claim: Hope Chinwo's left hip pain was associated with stiffness.
Label: UNSUPPORTED
Reason: The evidence mentions limited ROM and inability to fully bear weight, but does not explicitly state the pain was associated with stiffness.
--------------------------------------------------------------------------------
Claim ID: 29
Claim: Mild anaemia had been optimised following ferrous sulfate.
Label: PARTIALLY_SUPPORTED
Reason: The record supports that 

In [21]:
# ============================================================
# EXTRACT ALL NON-FULLY-SUPPORTED WORKFLOW 4 CLAIMS
# ============================================================

flagged_claim_rows = []

for person_id, patient_result in verification_results.items():

    for source_group in patient_result["source_groups"].values():

        for claim in source_group["verified_claims"]:

            if claim["label"] != "SUPPORTED":

                flagged_claim_rows.append({
                    "person_id": person_id,
                    "claim_id": claim["claim_id"],
                    "source_id": claim["source_id"],
                    "claim": claim["claim"],
                    "label": claim["label"],
                    "reason": claim["reason"],
                })

flagged_claims = pd.DataFrame(flagged_claim_rows)

print("Total flagged claims:", len(flagged_claims))
print("Affected patients:", flagged_claims["person_id"].nunique())

print("\nLabel counts:")
display(
    flagged_claims["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

display(flagged_claims.head(10))

Total flagged claims: 124
Affected patients: 35

Label counts:


,label,count
0,UNSUPPORTED,79
1,PARTIALLY_SUPPORTED,45


,person_id,claim_id,source_id,claim,label,reason
0,05192757-942f-460d-b4ff-004ec39cc5ee,3,1,Hope Chinwo's left hip pain was gradually wors...,UNSUPPORTED,The record supports severe left hip OA and lim...
1,05192757-942f-460d-b4ff-004ec39cc5ee,5,1,Hope Chinwo's left hip pain was associated wit...,UNSUPPORTED,The evidence mentions limited ROM and inabilit...
2,05192757-942f-460d-b4ff-004ec39cc5ee,29,8,Mild anaemia had been optimised following ferr...,PARTIALLY_SUPPORTED,The record supports that mild anaemia was pres...
3,05192757-942f-460d-b4ff-004ec39cc5ee,40,11,Estimated blood loss was approximately 150 mL.,UNSUPPORTED,The record does not state an estimated blood l...
4,05192757-942f-460d-b4ff-004ec39cc5ee,41,11,There were no intraoperative complications.,UNSUPPORTED,The record does not explicitly state that ther...
5,05192757-942f-460d-b4ff-004ec39cc5ee,113,28,Rivaroxaban was prescribed for postoperative a...,UNSUPPORTED,"Evidence 2 gives rivaroxaban for prophylaxis, ..."
6,05192757-942f-460d-b4ff-004ec39cc5ee,120,28,The intended discharge plan included follow-up...,UNSUPPORTED,"The evidence mentions follow-up physio, but do..."
7,05192757-942f-460d-b4ff-004ec39cc5ee,122,29,Her daughter received postoperative care.,UNSUPPORTED,The evidence mentions contact with the next of...
8,05192757-942f-460d-b4ff-004ec39cc5ee,124,29,Her daughter received hip-precaution instructi...,UNSUPPORTED,The record does not state that the daughter re...
9,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,23,4,Ferrous sulfate was prescribed for two weeks.,PARTIALLY_SUPPORTED,"Ferrous sulfate was prescribed, but the origin..."


In [22]:
# ============================================================
# RELATE VERIFICATION INTERVENTIONS TO COMPLETENESS LOSS
# ============================================================

# Count flagged verification claims per patient
flagged_per_patient = (
    flagged_claims
    .groupby(["person_id", "label"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure both label columns exist
for column in ["UNSUPPORTED", "PARTIALLY_SUPPORTED"]:
    if column not in flagged_per_patient.columns:
        flagged_per_patient[column] = 0

flagged_per_patient["total_flagged"] = (
    flagged_per_patient["UNSUPPORTED"]
    + flagged_per_patient["PARTIALLY_SUPPORTED"]
)

# Count completeness losses per affected patient
losses_per_patient = (
    verification_loss_details
    .groupby("person_id")
    .agg(
        completeness_facts_degraded=("fact_id", "count"),
        completeness_points_lost=("score_change", lambda x: -x.sum()),
    )
    .reset_index()
)

# Join the two
loss_vs_flags = losses_per_patient.merge(
    flagged_per_patient,
    on="person_id",
    how="left",
)

display(
    loss_vs_flags[
        [
            "person_id",
            "completeness_facts_degraded",
            "completeness_points_lost",
            "UNSUPPORTED",
            "PARTIALLY_SUPPORTED",
            "total_flagged",
        ]
    ].sort_values(
        "completeness_points_lost",
        ascending=False,
    )
)

print("\nAffected patients:", len(loss_vs_flags))

print(
    "Affected patients with verification flags:",
    (loss_vs_flags["total_flagged"] > 0).sum()
)

print(
    "Total completeness points lost:",
    loss_vs_flags["completeness_points_lost"].sum()
)

,person_id,completeness_facts_degraded,completeness_points_lost,UNSUPPORTED,PARTIALLY_SUPPORTED,total_flagged
0,05192757-942f-460d-b4ff-004ec39cc5ee,5,5,8.0,1.0,9.0
12,c332ce87-7afd-4a74-9896-3fef50179bd0,3,3,NaN,NaN,NaN
13,ce0046dc-0ad3-4710-8147-549793c58b44,2,3,3.0,2.0,5.0
14,f44d4a08-4c76-4000-ad61-d1d81032e643,3,3,4.0,2.0,6.0
1,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,2,2,2.0,2.0,4.0
3,31f9612b-5a6b-48ea-887b-895772a83b99,1,2,1.0,0.0,1.0
7,58b8aad6-7327-4450-956f-b775be0f4984,2,2,1.0,4.0,5.0
8,5e434d78-b2f6-4d88-b327-fff6ee50b901,2,2,6.0,1.0,7.0
15,fc6268ab-ad15-4613-9d8d-12c045e2fd46,1,2,1.0,2.0,3.0
2,29ea304f-821d-474e-81a1-394ca3945e02,1,1,3.0,1.0,4.0



Affected patients: 16
Affected patients with verification flags: 14
Total completeness points lost: 31


In [23]:
# ============================================================
# INVESTIGATE AFFECTED PATIENTS MISSING FROM FLAGGED-CLAIM JOIN
# ============================================================

missing_flag_patients = loss_vs_flags[
    loss_vs_flags["total_flagged"].isna()
]["person_id"].tolist()

print("Patients missing from flagged-claim join:")
print(missing_flag_patients)

for person_id in missing_flag_patients:

    print("\n" + "=" * 80)
    print("PATIENT:", person_id)

    print(
        "Exists in verification_results:",
        person_id in verification_results
    )

    if person_id in verification_results:
        patient_result = verification_results[person_id]

        labels = []

        for source_group in patient_result["source_groups"].values():
            for claim in source_group["verified_claims"]:
                labels.append(claim["label"])

        print("Verification labels:")
        print(pd.Series(labels).value_counts())

    print(
        "Revision needed:",
        verified_summaries[person_id]["revision_needed"]
    )

    print(
        "Issue count:",
        verified_summaries[person_id]["issue_count"]
    )

    print(
        "RAG summary identical to final summary:",
        rag_summaries[person_id]["summary"]
        == verified_summaries[person_id]["final_summary"]
    )

Patients missing from flagged-claim join:
['6e93f9d9-213d-4f2c-a1f0-f475dacef554', 'c332ce87-7afd-4a74-9896-3fef50179bd0']

PATIENT: 6e93f9d9-213d-4f2c-a1f0-f475dacef554
Exists in verification_results: True
Verification labels:
SUPPORTED    120
Name: count, dtype: int64
Revision needed: False
Issue count: 0
RAG summary identical to final summary: True

PATIENT: c332ce87-7afd-4a74-9896-3fef50179bd0
Exists in verification_results: True
Verification labels:
SUPPORTED    106
Name: count, dtype: int64
Revision needed: False
Issue count: 0
RAG summary identical to final summary: True


In [24]:
# ============================================================
# SEPARATE TRUE SUMMARY CHANGES FROM EVALUATOR INCONSISTENCY
# ============================================================

verification_loss_details = verification_loss_details.copy()

# Check whether Workflow 3 and Workflow 4 summaries are
# literally identical for every completeness-loss case.
verification_loss_details["summary_changed"] = (
    verification_loss_details["person_id"].apply(
        lambda person_id:
            rag_summaries[person_id]["summary"]
            != verified_summaries[person_id]["final_summary"]
    )
)

true_revision_losses = verification_loss_details[
    verification_loss_details["summary_changed"]
].copy()

identical_summary_losses = verification_loss_details[
    ~verification_loss_details["summary_changed"]
].copy()

print("All observed completeness losses:", len(verification_loss_details))

print(
    "Losses where summary actually changed:",
    len(true_revision_losses)
)

print(
    "Losses despite identical summaries:",
    len(identical_summary_losses)
)

print(
    "\nCompleteness points lost where summary changed:",
    -true_revision_losses["score_change"].sum()
)

print(
    "Completeness points lost despite identical summaries:",
    -identical_summary_losses["score_change"].sum()
)

print("\nIdentical-summary inconsistencies:")
display(
    identical_summary_losses[
        [
            "person_id",
            "fact_id",
            "fact",
            "rag_score",
            "verified_score",
            "score_change",
        ]
    ]
)

All observed completeness losses: 28
Losses where summary actually changed: 24
Losses despite identical summaries: 4

Completeness points lost where summary changed: 27
Completeness points lost despite identical summaries: 4

Identical-summary inconsistencies:


,person_id,fact_id,fact,rag_score,verified_score,score_change
16,6e93f9d9-213d-4f2c-a1f0-f475dacef554,15,Occupational therapy focused on safe transfers...,2,1,-1
19,c332ce87-7afd-4a74-9896-3fef50179bd0,10,The patient remained on the respiratory ward f...,2,1,-1
20,c332ce87-7afd-4a74-9896-3fef50179bd0,14,Inflammatory markers improved substantially du...,2,1,-1
21,c332ce87-7afd-4a74-9896-3fef50179bd0,18,He was discharged with advice to complete oral...,2,1,-1


In [25]:
# ============================================================
# BUILD BEFORE/AFTER TEXT DIFFS FOR TRUE REVISION LOSSES
# ============================================================

import difflib
import re


def split_summary_sentences(text):
    """Split summary text into readable sentence-like units."""
    text = str(text).replace("\n", " ")

    return [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", text)
        if sentence.strip()
    ]


def get_changed_text(original, revised):
    """
    Return only sentence-level text that was removed or added
    during Workflow 4 revision.
    """
    original_sentences = split_summary_sentences(original)
    revised_sentences = split_summary_sentences(revised)

    diff = difflib.ndiff(original_sentences, revised_sentences)

    removed = []
    added = []

    for line in diff:
        if line.startswith("- "):
            removed.append(line[2:])
        elif line.startswith("+ "):
            added.append(line[2:])

    return " ".join(removed), " ".join(added)


revision_analysis_rows = []

for _, row in true_revision_losses.iterrows():

    person_id = row["person_id"]

    original_summary = rag_summaries[person_id]["summary"]
    revised_summary = verified_summaries[person_id]["final_summary"]

    removed_text, added_text = get_changed_text(
        original_summary,
        revised_summary,
    )

    revision_analysis_rows.append({
        "person_id": person_id,
        "fact_id": row["fact_id"],
        "reference_fact": row["fact"],
        "rag_score": row["rag_score"],
        "verified_score": row["verified_score"],
        "removed_text": removed_text,
        "added_text": added_text,
    })


revision_loss_analysis = pd.DataFrame(revision_analysis_rows)

print("True revision-related loss cases:", len(revision_loss_analysis))

display(
    revision_loss_analysis[
        [
            "person_id",
            "fact_id",
            "reference_fact",
            "removed_text",
            "added_text",
        ]
    ]
)

True revision-related loss cases: 24


,person_id,fact_id,reference_fact,removed_text,added_text
0,05192757-942f-460d-b4ff-004ec39cc5ee,3,Pre-operative assessment showed mild anaemia w...,- **21/12/2025 – Preoperative assessment:** Ho...,- **21/12/2025 – Preoperative assessment:** Ho...
1,05192757-942f-460d-b4ff-004ec39cc5ee,8,The removed left femoral head was sent for his...,- **21/12/2025 – Preoperative assessment:** Ho...,- **21/12/2025 – Preoperative assessment:** Ho...
2,05192757-942f-460d-b4ff-004ec39cc5ee,10,Estimated intraoperative blood loss was approx...,- **21/12/2025 – Preoperative assessment:** Ho...,- **21/12/2025 – Preoperative assessment:** Ho...
3,05192757-942f-460d-b4ff-004ec39cc5ee,14,"Postoperative anaemia developed after surgery,...",- **21/12/2025 – Preoperative assessment:** Ho...,- **21/12/2025 – Preoperative assessment:** Ho...
4,05192757-942f-460d-b4ff-004ec39cc5ee,20,Discharge medications included rivaroxaban 10 ...,- **21/12/2025 – Preoperative assessment:** Ho...,- **21/12/2025 – Preoperative assessment:** Ho...
5,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,12,The patient mobilised progressively after surg...,Ferrous sulfate 200 mg daily and omeprazole 20...,Ferrous sulfate 200 mg daily was prescribed an...
6,0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf,13,Occupational therapy identified the need for h...,Ferrous sulfate 200 mg daily and omeprazole 20...,Ferrous sulfate 200 mg daily was prescribed an...
7,29ea304f-821d-474e-81a1-394ca3945e02,7,Repeat chest imaging showed gradual improvemen...,Final chest X-ray review on 09/01 showed compl...,"On 08/01, chest X-ray showed stable pneumomedi..."
8,31f9612b-5a6b-48ea-887b-895772a83b99,20,Occupational therapy recommended crutches and ...,Written and verbal advice was provided regardi...,Verbal advice was provided regarding wound car...
9,359014a1-10e6-4bd8-9ba7-513d021c971e,8,Serial chest radiographs showed progressive im...,He was considered clinically stable for discha...,Follow-up chest X-ray planned.


In [26]:
# ============================================================
# COMPACT AUDIT VIEW FOR THE 24 TRUE REVISION-LOSS CASES
# ============================================================

def show_revision_loss_case(case_number):
    """Display one completeness-loss case with its verification context."""

    row = revision_loss_analysis.iloc[case_number]
    person_id = row["person_id"]

    print("=" * 100)
    print(f"CASE {case_number + 1} / {len(revision_loss_analysis)}")
    print("PATIENT:", person_id)
    print("FACT ID:", row["fact_id"])

    print("\nREFERENCE FACT:")
    print(row["reference_fact"])

    print(
        f"\nCOMPLETENESS SCORE: "
        f"{row['rag_score']} → {row['verified_score']}"
    )

    print("\nTEXT REMOVED / CHANGED FROM RAG:")
    print(row["removed_text"])

    print("\nTEXT ADDED / CHANGED IN VERIFIED SUMMARY:")
    print(row["added_text"])

    print("\nVERIFIER FLAGS FOR THIS PATIENT:")

    patient_flags = flagged_claims[
        flagged_claims["person_id"] == person_id
    ]

    if len(patient_flags) == 0:
        print("None")
    else:
        for _, flag in patient_flags.iterrows():
            print(
                f"\n[{flag['label']}] "
                f"Claim {flag['claim_id']}: {flag['claim']}"
            )
            print("Reason:", flag["reason"])


# Start with the first case
show_revision_loss_case(0)

CASE 1 / 24
PATIENT: 05192757-942f-460d-b4ff-004ec39cc5ee
FACT ID: 3

REFERENCE FACT:
Pre-operative assessment showed mild anaemia with hemoglobin 11.3 g/dL, and ferrous sulfate was started for optimization before surgery.

COMPLETENESS SCORE: 2 → 1

TEXT REMOVED / CHANGED FROM RAG:
- **21/12/2025 – Preoperative assessment:** Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis, associated with stiffness and limited mobility. Mild anaemia had been optimised following ferrous sulfate, and elevated BP was considered consistent with known hypertension without acute concerns. Estimated blood loss was approximately 150 mL, with no intraoperative complications. The intended discharge plan included rivaroxaban 10 mg once daily for 7 days, ferrous sulfate 200 mg once daily for postoperative anaemia, a high-iron diet, outpatient physiotherapy, and follow-up appointments. She and her daughter received postoperative care and hip-precaution instru

In [27]:
# ============================================================
# MANUAL MECHANISM CLASSIFICATIONS
# ============================================================

mechanism_classifications = []

mechanism_classifications.append({
    "case": 1,
    "person_id": "05192757-942f-460d-b4ff-004ec39cc5ee",
    "fact_id": 3,
    "mechanism": "intended_evidence_constraining_revision",
    "notes": (
        "Verifier marked the claim that anaemia had been 'optimised' "
        "as PARTIALLY_SUPPORTED. Final revision weakened this to "
        "'being treated with ferrous sulfate', reducing completeness "
        "from 2 to 1."
    ),
})

pd.DataFrame(mechanism_classifications)

,case,person_id,fact_id,mechanism,notes
0,1,05192757-942f-460d-b4ff-004ec39cc5ee,3,intended_evidence_constraining_revision,Verifier marked the claim that anaemia had bee...


In [28]:
show_revision_loss_case(1)

CASE 2 / 24
PATIENT: 05192757-942f-460d-b4ff-004ec39cc5ee
FACT ID: 8

REFERENCE FACT:
The removed left femoral head was sent for histological examination.

COMPLETENESS SCORE: 2 → 1

TEXT REMOVED / CHANGED FROM RAG:
- **21/12/2025 – Preoperative assessment:** Hope Chinwo was evaluated for severe, gradually worsening left hip pain due to end-stage osteoarthritis, associated with stiffness and limited mobility. Mild anaemia had been optimised following ferrous sulfate, and elevated BP was considered consistent with known hypertension without acute concerns. Estimated blood loss was approximately 150 mL, with no intraoperative complications. The intended discharge plan included rivaroxaban 10 mg once daily for 7 days, ferrous sulfate 200 mg once daily for postoperative anaemia, a high-iron diet, outpatient physiotherapy, and follow-up appointments. She and her daughter received postoperative care and hip-precaution instructions.

TEXT ADDED / CHANGED IN VERIFIED SUMMARY:
- **21/12/2025 – 

In [29]:
## RAG Omission Analysis

# This analysis investigates the lower completeness observed for the RAG workflow
# by determining whether clinically important facts omitted from the final RAG
# summary were:

# 1. absent from the Top-20 retrieved context (retrieval omission), or
# 2. present in the retrieved context but omitted during summarization
#    (generation/synthesis omission).

In [31]:
# ============================================================
# RECONSTRUCT EXACT WORKFLOW-3 RAG TOP-20 RETRIEVAL
# ============================================================

import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# 1. Reproduce the exact production section chunker
# ------------------------------------------------------------

def split_note_into_sections(note_text):
    """Split a clinical note using recognized section headings."""

    text = str(note_text).strip()

    if not text:
        return []

    section_names = [
        "Chief Complaint",
        "Presenting Complaint",
        "History of Present Illness",
        "HPI",
        "Past Medical History",
        "PMH",
        "Past Surgical History",
        "PSH",
        "Medications",
        "Current Medications",
        "Allergies",
        "Family History",
        "Social History",
        "Review of Systems",
        "ROS",
        "Physical Examination",
        "Physical Exam",
        "Examination",
        "Vital Signs",
        "Vitals",
        "Investigations",
        "Laboratory Results",
        "Labs",
        "Imaging",
        "Assessment",
        "Impression",
        "Diagnosis",
        "Diagnoses",
        "Plan",
        "Assessment and Plan",
        "Treatment",
        "Hospital Course",
        "Clinical Course",
        "Discharge Plan",
        "Follow Up",
        "Follow-Up",
    ]

    heading_pattern = "|".join(
        re.escape(name)
        for name in sorted(
            section_names,
            key=len,
            reverse=True,
        )
    )

    pattern = re.compile(
        rf"(?im)^[ \t]*(?P<heading>{heading_pattern})"
        rf"[ \t]*(?::|-)?[ \t]*$"
    )

    matches = list(pattern.finditer(text))

    if not matches:
        return [{
            "section_name": "Unsectioned",
            "chunk_text": text,
        }]

    sections = []

    # Preserve text before first recognized heading.
    prefix = text[:matches[0].start()].strip()

    if prefix:
        sections.append({
            "section_name": "Preamble",
            "chunk_text": prefix,
        })

    # Extract recognized sections.
    for i, match in enumerate(matches):

        section_name = match.group("heading").strip()
        content_start = match.end()

        if i + 1 < len(matches):
            content_end = matches[i + 1].start()
        else:
            content_end = len(text)

        content = text[content_start:content_end].strip()

        if content:
            sections.append({
                "section_name": section_name,
                "chunk_text": f"{section_name}\n{content}",
            })

    return sections


# ------------------------------------------------------------
# 2. Reproduce frozen source preprocessing
# ------------------------------------------------------------

clinical_notes = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "clinical_notes.csv"
)

notes_clean = clinical_notes[
    clinical_notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values([
        "person_id",
        "creation_timestamp",
    ])
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Recreate exact production section chunks
# ------------------------------------------------------------

section_records = []

for _, row in notes_dedup.iterrows():

    sections = split_note_into_sections(
        row["clean_note_text"]
    )

    for section in sections:
        section_records.append({
            "person_id": row["person_id"],
            "creation_timestamp": row["creation_timestamp"],
            "section_name": section["section_name"],
            "chunk_text": section["chunk_text"],
        })

section_chunks = pd.DataFrame(section_records)

section_chunks = (
    section_chunks
    .sort_values([
        "person_id",
        "creation_timestamp",
    ])
    .reset_index(drop=True)
)

section_chunks["chunk_id"] = range(
    len(section_chunks)
)

print("Reconstructed section chunks:", len(section_chunks))


# ------------------------------------------------------------
# 4. Recreate BGE embeddings
# ------------------------------------------------------------

bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

section_embeddings = bge_model.encode(
    section_chunks["chunk_text"].astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)


# ------------------------------------------------------------
# 5. Frozen production retrieval query
# ------------------------------------------------------------

RAG_QUERY = """
Retrieve the clinically relevant information needed to produce a
comprehensive longitudinal summary of this patient's clinical history,
including major diagnoses, treatments, investigations, clinical
progression, and outcomes.
""".strip()

query_embedding = bge_model.encode(
    [RAG_QUERY],
    normalize_embeddings=True,
)


# ------------------------------------------------------------
# 6. Reconstruct Top-20 retrieval for all 50 study patients
# ------------------------------------------------------------

with open(
    PROJECT_ROOT / "data" / "raw" / "patients" / "study_patient_ids.json",
    "r",
    encoding="utf-8",
) as f:
    study_patient_ids = json.load(f)


rag_top20_retrievals = {}

for patient_id in study_patient_ids:

    patient_mask = (
        section_chunks["person_id"] == patient_id
    )

    patient_chunks = (
        section_chunks.loc[patient_mask]
        .copy()
    )

    patient_embeddings = section_embeddings[
        patient_mask.to_numpy()
    ]

    similarities = cosine_similarity(
        query_embedding,
        patient_embeddings,
    )[0]

    patient_chunks["similarity"] = similarities

    top_k = min(
        20,
        len(patient_chunks),
    )

    retrieved = (
        patient_chunks
        .sort_values(
            "similarity",
            ascending=False,
        )
        .head(top_k)
        .copy()
    )

    retrieved["retrieval_rank"] = range(
        1,
        len(retrieved) + 1,
    )

    # Same chronological reorder used before summarization.
    retrieved = (
        retrieved
        .sort_values([
            "creation_timestamp",
            "chunk_id",
        ])
        .reset_index(drop=True)
    )

    rag_top20_retrievals[patient_id] = retrieved


print("Patients reconstructed:", len(rag_top20_retrievals))

print(
    "Total retrieved chunks:",
    sum(len(x) for x in rag_top20_retrievals.values()),
)

Reconstructed section chunks: 2771


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Patients reconstructed: 50
Total retrieved chunks: 1000


In [32]:
# ============================================================
# IDENTIFY COMPLETELY OMITTED CLINICAL FACTS IN RAG
# ============================================================

rag_missed_facts = []

for person_id, result in completeness_results.items():

    for fact_score in result["scores"]:

        # Score 0 = clinically important reference fact was
        # completely absent from the final RAG summary.
        if fact_score["rag"] == 0:

            fact_id = fact_score["fact_id"]

            # Get the corresponding reference fact text/category.
            reference_fact = next(
                fact
                for fact in completeness_references[person_id]["facts"]
                if fact["fact_id"] == fact_id
            )

            rag_missed_facts.append({
                "person_id": person_id,
                "fact_id": fact_id,
                "category": reference_fact["category"],
                "fact": reference_fact["fact"],
            })


rag_missed_facts = pd.DataFrame(rag_missed_facts)

print(
    "Completely omitted RAG reference facts:",
    len(rag_missed_facts)
)

print(
    "Patients with at least one complete omission:",
    rag_missed_facts["person_id"].nunique()
)

print("\nOmissions by clinical category:")
display(
    rag_missed_facts["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

Completely omitted RAG reference facts: 41
Patients with at least one complete omission: 18

Omissions by clinical category:


,category,count
0,investigation,15
1,treatment,8
2,procedure,7
3,outcome,6
4,diagnosis,2
5,clinical_course,2
6,complication,1


In [33]:
# ============================================================
# ATTACH WORKFLOW-3 TOP-20 CONTEXT TO EACH MISSED RAG FACT
# ============================================================

rag_omission_audit = []

for _, row in rag_missed_facts.iterrows():

    person_id = row["person_id"]

    retrieved = rag_top20_retrievals[person_id]

    # Combine exactly the chunks that were available to
    # the RAG summarization model.
    retrieved_context = "\n\n".join(
        retrieved["chunk_text"]
        .astype(str)
        .tolist()
    )

    rag_omission_audit.append({
        "person_id": person_id,
        "fact_id": row["fact_id"],
        "category": row["category"],
        "fact": row["fact"],
        "retrieved_chunk_count": len(retrieved),
        "retrieved_context": retrieved_context,
    })


rag_omission_audit = pd.DataFrame(
    rag_omission_audit
)

print("Missed facts prepared for audit:", len(rag_omission_audit))
print(
    "Affected patients:",
    rag_omission_audit["person_id"].nunique()
)

display(
    rag_omission_audit[
        [
            "person_id",
            "fact_id",
            "category",
            "fact",
        ]
    ].head(10)
)

Missed facts prepared for audit: 41
Affected patients: 18


,person_id,fact_id,category,fact
0,04df53ea-55c1-48d9-84a1-1f15c133b29b,4,diagnosis,Mild hyponatremia was identified during admiss...
1,04df53ea-55c1-48d9-84a1-1f15c133b29b,6,treatment,Hyponatremia was treated with oral sodium chlo...
2,04df53ea-55c1-48d9-84a1-1f15c133b29b,7,treatment,Headache after the fall was treated with parac...
3,04df53ea-55c1-48d9-84a1-1f15c133b29b,8,investigation,Repeat CT head on 03/01/26 showed no progressi...
4,136c7916-4f9b-4e5c-bf01-77e9d2c681a2,2,investigation,CT head showed a subdural hygroma with no midl...
5,51f15281-8840-4fd0-92de-89188ab8d736,9,outcome,The operation had minimal blood loss and no in...
6,5e434d78-b2f6-4d88-b327-fff6ee50b901,8,diagnosis,An acute asthma exacerbation was also consider...
7,61699d6d-904a-4ece-9026-cd61b3fd9a50,5,investigation,Inflammatory markers were not significantly el...
8,61699d6d-904a-4ece-9026-cd61b3fd9a50,10,clinical_course,The patient had mild subcutaneous emphysema in...
9,69bf7e25-abb2-4dde-857f-f1138d4d0d8a,3,investigation,MRI head later confirmed the subdural hygroma ...


In [34]:
# ============================================================
# FIND BEST RETRIEVED CHUNK FOR EACH MISSED REFERENCE FACT
# ============================================================

audit_rows = []

for _, row in rag_omission_audit.iterrows():

    person_id = row["person_id"]
    fact = row["fact"]

    retrieved = rag_top20_retrievals[person_id].copy()

    # Embed the missed clinical fact.
    fact_embedding = bge_model.encode(
        [fact],
        normalize_embeddings=True,
    )

    # Embed this patient's exact Top-20 retrieved chunks.
    chunk_embeddings = bge_model.encode(
        retrieved["chunk_text"].astype(str).tolist(),
        normalize_embeddings=True,
    )

    similarities = cosine_similarity(
        fact_embedding,
        chunk_embeddings,
    )[0]

    best_index = int(np.argmax(similarities))
    best_chunk = retrieved.iloc[best_index]

    audit_rows.append({
        "person_id": person_id,
        "fact_id": row["fact_id"],
        "category": row["category"],
        "fact": fact,
        "best_similarity": float(similarities[best_index]),
        "best_chunk": best_chunk["chunk_text"],
    })


rag_omission_matches = pd.DataFrame(audit_rows)

print("Facts matched:", len(rag_omission_matches))

display(
    rag_omission_matches[
        [
            "fact_id",
            "category",
            "fact",
            "best_similarity",
            "best_chunk",
        ]
    ]
)

Facts matched: 41


,fact_id,category,fact,best_similarity,best_chunk
0,4,diagnosis,Mild hyponatremia was identified during admiss...,0.628888,"- Patient: Allan Victor R obinson, 67-year-old..."
1,6,treatment,Hyponatremia was treated with oral sodium chlo...,0.590863,"10:15, 01/01/26 - Patient Allan Victor Robinso..."
2,7,treatment,Headache after the fall was treated with parac...,0.614571,Physiotherapy assessment conducted on 01/01/26...
3,8,investigation,Repeat CT head on 03/01/26 showed no progressi...,0.695391,"10:15, 01/01/26 - Patient Allan Victor Robinso..."
4,2,investigation,CT head showed a subdural hygroma with no midl...,0.666040,Neurologicakl observations conducted by Nurse ...
5,9,outcome,The operation had minimal blood loss and no in...,0.594912,"Patient: Mr. Bertram Adam Richards, Male, 43 y..."
6,8,diagnosis,An acute asthma exacerbation was also consider...,0.663910,"03:35, 06/01/26. Triage by Nurse Mandikudza Ny..."
7,5,investigation,Inflammatory markers were not significantly el...,0.601885,"Patient Jonathan Edward Hodgson, 40M, presents..."
8,10,clinical_course,The patient had mild subcutaneous emphysema in...,0.689575,Therapist Manjeet Das (Physical) reviewed the ...
9,3,investigation,MRI head later confirmed the subdural hygroma ...,0.621261,Patient escorted to radiology by Nurse Kelly N...
